# Transformer OASIS - RMIA Black-Box Attack

This notebook is a standalone RMIA attack notebook created in the project root.
It reuses the same transformer architecture and prepared OASIS data pipeline.

What is implemented:
- get_true_class_confidence
- build_signal_matrix
- get_rmia_out_signals
- run_rmia
- tune_offline_a
- compute_attack_results
- audit_target_model
- run_mia_transformer(config)

In [14]:
import os
import random
from dataclasses import dataclass
from typing import Any, Dict, Iterable, List, Sequence, Tuple

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import roc_curve, auc
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, Model

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

def set_global_determinism(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def build_transformer(n_features: int, dropout: float = 0.0) -> tf.keras.Model:
    inp = layers.Input(shape=(1, n_features))
    x = layers.Dense(64)(inp)
    a = layers.MultiHeadAttention(num_heads=4, key_dim=16, dropout=dropout)(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + a)
    f = layers.Dense(128, activation='relu')(x)
    f = layers.Dropout(dropout)(f)
    f = layers.Dense(64)(f)
    x = layers.LayerNormalization(epsilon=1e-6)(x + f)
    x = layers.Flatten()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(2, activation='softmax')(x)

    model = Model(inp, out)
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

SEED = 42
set_global_determinism(SEED)
print('Environment ready')

Environment ready


In [15]:
prepared_path = os.path.join('data_preparation', 'prepared_oasis2_transformer.npz')
if not os.path.exists(prepared_path):
    prepared_path = os.path.join('..', 'data_preparation', 'prepared_oasis2_transformer.npz')
if not os.path.exists(prepared_path):
    raise FileNotFoundError('prepared_oasis2_transformer.npz not found')

b = np.load(prepared_path, allow_pickle=True)
X_train = b['X_train'].astype(np.float32)
X_test = b['X_test'].astype(np.float32)
y_train = b['y_train'].astype(np.int32)
y_test = b['y_test'].astype(np.int32)

if 'X_shadow_raw' in b:
    X_shadow_raw = b['X_shadow_raw'].astype(np.float32)
else:
    X_shadow_raw = b['X_shadow'].astype(np.float32)
y_shadow = b['y_shadow'].astype(np.int32)

X_train_seq = X_train.reshape(-1, 1, X_train.shape[1]).astype(np.float32)
X_test_seq = X_test.reshape(-1, 1, X_test.shape[1]).astype(np.float32)
X_shadow_seq = X_shadow_raw.reshape(-1, 1, X_shadow_raw.shape[1]).astype(np.float32)

print('=== DATA SOURCE ===')
print(f'Prepared file used: {os.path.abspath(prepared_path)}')
print(f'Loaded train={X_train_seq.shape}, test={X_test_seq.shape}, shadow={X_shadow_seq.shape}')
print(f'Class ratio train={y_train.mean():.4f}, test={y_test.mean():.4f}')

=== DATA SOURCE ===
Prepared file used: c:\Users\khalil\Desktop\stage_pfe\mia-defense-evaluation\data_preparation\prepared_oasis2_transformer.npz
Loaded train=(70, 1, 9), test=(284, 1, 9), shadow=(107, 1, 9)
Class ratio train=0.3571, test=0.3592


In [16]:
def make_numpy_dataloader(x: np.ndarray, y: np.ndarray, batch_size: int = 64) -> List[Tuple[np.ndarray, np.ndarray]]:
    batches = []
    for i in range(0, len(y), batch_size):
        batches.append((x[i:i + batch_size], y[i:i + batch_size]))
    return batches


def _ensure_memberships_shape(all_memberships: np.ndarray, num_samples: int, num_models: int) -> np.ndarray:
    m = np.asarray(all_memberships)
    if m.dtype != bool:
        m = m.astype(bool)

    if m.shape == (num_samples, num_models):
        return m
    if m.shape == (num_models, num_samples):
        return m.T
    raise ValueError(
        f'all_memberships must be (num_samples, num_models) or (num_models, num_samples). Got {m.shape}. '
        f'Expected samples={num_samples}, models={num_models}.'
    )


def get_true_class_confidence(model: tf.keras.Model, dataloader: Iterable[Tuple[np.ndarray, np.ndarray]], device: str = 'cpu') -> np.ndarray:
    # Keras uses configured backend device; keep arg for API compatibility.
    _ = device
    confs = []
    for xb, yb in dataloader:
        probs = np.asarray(model.predict(xb, verbose=0))
        if probs.ndim == 1:
            probs = np.column_stack([1.0 - probs, probs])
        if probs.ndim != 2:
            raise ValueError(f'Expected probs shape (batch, num_classes), got {probs.shape}')

        yb = np.asarray(yb).reshape(-1).astype(np.int64)
        if probs.shape[0] != len(yb):
            raise ValueError('Batch size mismatch between outputs and labels')
        if (yb < 0).any() or (yb >= probs.shape[1]).any():
            raise ValueError('Label index out of range for model output classes')

        conf = probs[np.arange(len(yb)), yb]
        confs.append(conf.reshape(-1, 1))

    if not confs:
        raise ValueError('Dataloader is empty')
    return np.concatenate(confs, axis=0).reshape(-1)


def build_signal_matrix(models: Sequence[tf.keras.Model], dataloader: Iterable[Tuple[np.ndarray, np.ndarray]], device: str = 'cpu') -> np.ndarray:
    if len(models) == 0:
        raise ValueError('models must contain at least one model')
    cols = [get_true_class_confidence(m, dataloader, device=device).reshape(-1, 1) for m in models]
    lengths = [c.shape[0] for c in cols]
    if len(set(lengths)) != 1:
        raise ValueError(f'Signal length mismatch across models: {lengths}')
    return np.concatenate(cols, axis=1)


def get_rmia_out_signals(all_signals: np.ndarray, all_memberships: np.ndarray, target_model_idx: int, num_reference_models: int) -> np.ndarray:
    if num_reference_models < 1:
        raise ValueError('num_reference_models must be >= 1')

    all_signals = np.asarray(all_signals, dtype=np.float64)
    all_memberships = np.asarray(all_memberships, dtype=bool)

    if all_signals.shape != all_memberships.shape:
        raise ValueError('all_signals and all_memberships must have same shape')

    n_models = all_signals.shape[1]
    if not (0 <= target_model_idx < n_models):
        raise ValueError(f'target_model_idx {target_model_idx} out of range [0, {n_models})')

    paired_model_idx = target_model_idx + 1 if target_model_idx % 2 == 0 else target_model_idx - 1
    columns = [i for i in range(n_models) if i != target_model_idx and i != paired_model_idx][:2 * num_reference_models]
    if len(columns) == 0:
        raise ValueError('No reference models available after excluding target and paired model')

    selected_signals = all_signals[:, columns]
    non_members = ~all_memberships[:, columns]
    out_signals = selected_signals * non_members
    out_signals = -np.sort(-out_signals, axis=1)[:, :num_reference_models]
    return out_signals


def run_rmia(target_model_idx: int, all_signals: np.ndarray, population_signals: np.ndarray, all_memberships: np.ndarray, num_reference_models: int, offline_a: float) -> np.ndarray:
    if not (0.0 <= offline_a <= 1.0):
        raise ValueError('offline_a must be in [0, 1]')

    target_signals = all_signals[:, target_model_idx]
    out_signals = get_rmia_out_signals(all_signals, all_memberships, target_model_idx, num_reference_models)
    mean_out_x = np.mean(out_signals, axis=1)
    mean_x = (1.0 + offline_a) / 2.0 * mean_out_x + (1.0 - offline_a) / 2.0
    mean_x = np.clip(mean_x, 1e-12, None)
    prob_ratio_x = target_signals.ravel() / mean_x

    z_signals = population_signals[:, target_model_idx]
    population_memberships = np.zeros_like(population_signals, dtype=bool)
    z_out_signals = get_rmia_out_signals(population_signals, population_memberships, target_model_idx, num_reference_models)
    mean_out_z = np.mean(z_out_signals, axis=1)
    mean_z = (1.0 + offline_a) / 2.0 * mean_out_z + (1.0 - offline_a) / 2.0
    mean_z = np.clip(mean_z, 1e-12, None)
    prob_ratio_z = z_signals.ravel() / mean_z

    ratios = prob_ratio_x[:, np.newaxis] / np.clip(prob_ratio_z, 1e-12, None)
    score = np.mean(ratios > 1.0, axis=1)
    return score


def tune_offline_a(target_model_idx: int, all_signals: np.ndarray, population_signals: np.ndarray, all_memberships: np.ndarray, num_reference_models: int = 1) -> Tuple[float, np.ndarray, np.ndarray]:
    paired_model_idx = target_model_idx + 1 if target_model_idx % 2 == 0 else target_model_idx - 1
    if not (0 <= paired_model_idx < all_signals.shape[1]):
        raise ValueError('Paired model index out of range')

    paired_memberships = all_memberships[:, paired_model_idx]

    best_a = 0.0
    max_auc = -1.0
    best_scores = None

    for test_a in np.arange(0.0, 1.01, 0.1):
        mia_scores = run_rmia(
            target_model_idx=paired_model_idx,
            all_signals=all_signals,
            population_signals=population_signals,
            all_memberships=all_memberships,
            num_reference_models=num_reference_models,
            offline_a=float(test_a),
        )
        fpr_list, tpr_list, _ = roc_curve(paired_memberships.ravel(), mia_scores.ravel())
        roc_auc = auc(fpr_list, tpr_list)

        if roc_auc > max_auc:
            max_auc = roc_auc
            best_a = float(test_a)
            best_scores = mia_scores.ravel().copy()

    if best_scores is None:
        raise RuntimeError('Failed to tune offline_a')

    return best_a, best_scores, paired_memberships.ravel().copy()


def tune_rmia_hyperparams(
    target_model_idx: int,
    all_signals: np.ndarray,
    population_signals: np.ndarray,
    all_memberships: np.ndarray,
    reference_candidates: Sequence[int],
) -> Tuple[float, int, float]:
    """Tune (offline_a, num_reference_models) on paired model only (realistic black-box calibration)."""
    best_auc = -1.0
    best_a = 0.0
    best_refs = 1

    for refs in reference_candidates:
        try:
            cand_a, scores, paired_mem = tune_offline_a(
                target_model_idx=target_model_idx,
                all_signals=all_signals,
                population_signals=population_signals,
                all_memberships=all_memberships,
                num_reference_models=int(refs),
            )
            fpr, tpr, _ = roc_curve(paired_mem.ravel(), scores.ravel())
            cand_auc = float(auc(fpr, tpr))
            if cand_auc > best_auc:
                best_auc = cand_auc
                best_a = cand_a
                best_refs = int(refs)
        except Exception:
            continue

    if best_auc < 0:
        raise RuntimeError('RMIA hyperparameter tuning failed for all candidates')
    return best_a, best_refs, best_auc


def compute_attack_results(mia_scores: np.ndarray, target_memberships: np.ndarray) -> Dict[str, Any]:
    scores = np.asarray(mia_scores).ravel()
    memberships = np.asarray(target_memberships).ravel().astype(bool)
    if scores.shape[0] != memberships.shape[0]:
        raise ValueError('mia_scores and target_memberships must have same length')

    fpr_list, tpr_list, thr = roc_curve(memberships, scores)
    roc_auc = auc(fpr_list, tpr_list)

    def tpr_at(max_fpr: float) -> float:
        idx = np.where(fpr_list <= max_fpr)[0]
        return float(tpr_list[idx[-1]]) if len(idx) > 0 else 0.0

    return {
        'fpr': fpr_list,
        'tpr': tpr_list,
        'thresholds': thr,
        'auc': float(roc_auc),
        'tpr_at_10pct_fpr': tpr_at(0.10),
        'tpr_at_5pct_fpr': tpr_at(0.05),
        'tpr_at_1pct_fpr': tpr_at(0.01),
        'tpr_at_0.1pct_fpr': tpr_at(0.001),
        'tpr_at_0pct_fpr': tpr_at(0.0),
    }


@dataclass
class RMIAAuditResult:
    target_model_idx: int
    offline_a: float
    num_reference_models: int
    paired_tuning_auc: float
    mia_scores: np.ndarray
    memberships: np.ndarray
    attack_results: Dict[str, Any]


def audit_target_model(target_model_idx: int, all_signals: np.ndarray, population_signals: np.ndarray, all_memberships: np.ndarray, num_reference_models: int, auto_tune_reference_models: bool = False, reference_candidates: Sequence[int] = (1, 2, 3)) -> RMIAAuditResult:
    if auto_tune_reference_models:
        offline_a, tuned_refs, tune_auc = tune_rmia_hyperparams(
            target_model_idx=target_model_idx,
            all_signals=all_signals,
            population_signals=population_signals,
            all_memberships=all_memberships,
            reference_candidates=reference_candidates,
        )
        num_reference_models = tuned_refs
    else:
        offline_a, _, _ = tune_offline_a(
            target_model_idx=target_model_idx,
            all_signals=all_signals,
            population_signals=population_signals,
            all_memberships=all_memberships,
            num_reference_models=num_reference_models,
        )
        tune_auc = float('nan')

    mia_scores = run_rmia(
        target_model_idx=target_model_idx,
        all_signals=all_signals,
        population_signals=population_signals,
        all_memberships=all_memberships,
        num_reference_models=num_reference_models,
        offline_a=offline_a,
    )
    target_memberships = all_memberships[:, target_model_idx]
    attack_results = compute_attack_results(mia_scores, target_memberships)

    return RMIAAuditResult(
        target_model_idx=target_model_idx,
        offline_a=offline_a,
        num_reference_models=num_reference_models,
        paired_tuning_auc=tune_auc,
        mia_scores=mia_scores,
        memberships=target_memberships,
        attack_results=attack_results,
    )


def run_mia_transformer(config: Dict[str, Any]) -> Dict[str, Any]:
    target_model_idx = int(config['target_model_idx'])
    num_reference_models = int(config.get('num_reference_models', 2))
    auto_tune_reference_models = bool(config.get('auto_tune_reference_models', True))
    reference_candidates = config.get('reference_candidates', [1, 2, 3])

    models = list(config['models'])
    audit_dataloader = config['audit_dataloader']
    population_dataloader = config['population_dataloader']
    memberships_raw = np.asarray(config['memberships'])
    device = str(config.get('device', 'cpu'))

    if len(models) < 2:
        raise ValueError('Need at least target + paired model for RMIA')

    all_signals = build_signal_matrix(models, audit_dataloader, device=device)
    population_signals = build_signal_matrix(models, population_dataloader, device=device)

    all_memberships = _ensure_memberships_shape(
        memberships_raw,
        num_samples=all_signals.shape[0],
        num_models=all_signals.shape[1],
    )

    if population_signals.shape[1] != all_signals.shape[1]:
        raise ValueError('Population and audit signals must have same number of model columns')

    result = audit_target_model(
        target_model_idx=target_model_idx,
        all_signals=all_signals,
        population_signals=population_signals,
        all_memberships=all_memberships,
        num_reference_models=num_reference_models,
        auto_tune_reference_models=auto_tune_reference_models,
        reference_candidates=reference_candidates,
    )

    return {
        'target_model_idx': result.target_model_idx,
        'offline_a': result.offline_a,
        'num_reference_models': result.num_reference_models,
        'paired_tuning_auc': result.paired_tuning_auc,
        'auc': result.attack_results['auc'],
        'tpr_at_10pct_fpr': result.attack_results['tpr_at_10pct_fpr'],
        'tpr_at_5pct_fpr': result.attack_results['tpr_at_5pct_fpr'],
        'tpr_at_1pct_fpr': result.attack_results['tpr_at_1pct_fpr'],
        'tpr_at_0.1pct_fpr': result.attack_results['tpr_at_0.1pct_fpr'],
        'tpr_at_0pct_fpr': result.attack_results['tpr_at_0pct_fpr'],
        'mia_scores': result.mia_scores,
        'memberships': result.memberships,
        'attack_results': result.attack_results,
        'all_signals': all_signals,
        'population_signals': population_signals,
    }


print('RMIA functions loaded (enhanced black-box mode)')

RMIA functions loaded (enhanced black-box mode)


In [17]:
# Train or load target model + build shadow/reference models
TARGET_CHECKPOINT = os.path.join('transformer_pipeline', 'checkpoints', 'target_transformer.keras')
FORCE_RISKY_TRAINING_IF_NO_CHECKPOINT = True
TARGET_TRAIN_EPOCHS_RISKY = 260


def _model_eval_binary(model: tf.keras.Model, x_seq: np.ndarray, y_true: np.ndarray) -> Dict[str, float]:
    probs = model.predict(x_seq, verbose=0)
    if probs.ndim == 2 and probs.shape[1] == 2:
        p1 = probs[:, 1]
    elif probs.ndim == 2 and probs.shape[1] == 1:
        p1 = probs[:, 0]
    else:
        raise ValueError(f'Unexpected prediction shape: {probs.shape}')

    pred = (p1 >= 0.5).astype(np.int32)
    acc = float((pred == y_true).mean())

    try:
        from sklearn.metrics import roc_auc_score

        auc_val = float(roc_auc_score(y_true, p1))
    except Exception:
        auc_val = float('nan')

    return {
        'acc': acc,
        'auc': auc_val,
        'p_mean': float(np.mean(p1)),
        'p_std': float(np.std(p1)),
    }


target_model = build_transformer(n_features=X_train.shape[1], dropout=0.0)
if os.path.exists(TARGET_CHECKPOINT):
    target_model.load_weights(TARGET_CHECKPOINT)
    target_source = f'checkpoint: {os.path.abspath(TARGET_CHECKPOINT)}'
else:
    if FORCE_RISKY_TRAINING_IF_NO_CHECKPOINT:
        # Still black-box: we only make the target more overfit-prone, no white-box leakage used.
        target_model.fit(X_train_seq, y_train, epochs=TARGET_TRAIN_EPOCHS_RISKY, batch_size=32, verbose=0)
        target_source = f'trained in notebook (risky profile, epochs={TARGET_TRAIN_EPOCHS_RISKY})'
    else:
        target_model.fit(X_train_seq, y_train, epochs=120, batch_size=32, validation_split=0.2, verbose=0)
        target_source = 'trained in this notebook (safe profile)'

# Build audit pool: members + non-members for target model
X_audit = np.vstack([X_train_seq, X_test_seq]).astype(np.float32)
y_audit = np.concatenate([y_train, y_test]).astype(np.int32)
n_audit = len(y_audit)
target_membership = np.concatenate([np.ones(len(y_train), dtype=bool), np.zeros(len(y_test), dtype=bool)])

# Stronger but still realistic black-box attacker: more references
num_shadow_models = 11
shadow_epochs = 60
models = [target_model]
memberships_rows = [target_membership]

for s in range(num_shadow_models):
    set_global_determinism(SEED + 100 + s)
    idx = np.random.choice(n_audit, size=max(40, n_audit // 2), replace=False)
    m = np.zeros(n_audit, dtype=bool)
    m[idx] = True

    sh = build_transformer(n_features=X_train.shape[1], dropout=0.0)
    sh.fit(X_audit[idx], y_audit[idx], epochs=shadow_epochs, batch_size=32, verbose=0)

    models.append(sh)
    memberships_rows.append(m)

memberships_matrix = np.stack(memberships_rows, axis=0)

# Transformer target metrics
train_m = _model_eval_binary(target_model, X_train_seq, y_train)
test_m = _model_eval_binary(target_model, X_test_seq, y_test)
transformer_metrics = pd.DataFrame([
    {
        'model': 'target_transformer',
        'source': target_source,
        'train_acc': train_m['acc'],
        'test_acc': test_m['acc'],
        'gap_acc': train_m['acc'] - test_m['acc'],
        'train_auc': train_m['auc'],
        'test_auc': test_m['auc'],
        'test_p_mean': test_m['p_mean'],
        'test_p_std': test_m['p_std'],
        'num_shadow_models': num_shadow_models,
        'shadow_epochs': shadow_epochs,
    }
])

print('=== TRANSFORMER TARGET ===')
print(f'Model source: {target_source}')
print(f'Models ready: {len(models)} total (1 target + {num_shadow_models} shadows)')
display(transformer_metrics.round(4))

=== TRANSFORMER TARGET ===
Model source: trained in notebook (risky profile, epochs=260)
Models ready: 12 total (1 target + 11 shadows)


,model,source,train_acc,test_acc,gap_acc,train_auc,test_auc,test_p_mean,test_p_std,num_shadow_models,shadow_epochs
0,target_transformer,"trained in notebook (risky profile, epochs=260)",1.0,0.9296,0.0704,1.0,0.9893,0.3195,0.459,11,60


In [18]:
# Build dataloaders and run RMIA
audit_loader = make_numpy_dataloader(X_audit, y_audit, batch_size=64)
population_loader = make_numpy_dataloader(X_shadow_seq, y_shadow, batch_size=64)

config = {
    'target_model_idx': 0,
    'num_reference_models': 2,
    'auto_tune_reference_models': True,
    'reference_candidates': [1, 2, 3, 4],
    'models': models,
    'audit_dataloader': audit_loader,
    'population_dataloader': population_loader,
    'memberships': memberships_matrix,  # shape (num_models, num_samples) is accepted
    'device': 'cpu',
}

rmia_result = run_mia_transformer(config)

attack_summary = pd.DataFrame([
    {
        'attack': 'RMIA_blackbox_enhanced',
        'target_model_idx': rmia_result['target_model_idx'],
        'num_reference_models': rmia_result['num_reference_models'],
        'paired_tuning_auc': rmia_result['paired_tuning_auc'],
        'offline_a': rmia_result['offline_a'],
        'auc': rmia_result['auc'],
        'tpr_at_10pct_fpr': rmia_result['tpr_at_10pct_fpr'],
        'tpr_at_5pct_fpr': rmia_result['tpr_at_5pct_fpr'],
        'tpr_at_1pct_fpr': rmia_result['tpr_at_1pct_fpr'],
        'tpr_at_0.1pct_fpr': rmia_result['tpr_at_0.1pct_fpr'],
        'tpr_at_0pct_fpr': rmia_result['tpr_at_0pct_fpr'],
        'audit_samples': int(len(rmia_result['memberships'])),
        'population_samples': int(rmia_result['population_signals'].shape[0]),
    }
])

print('=== ATTACK RMIA RESULTS ===')
display(attack_summary.round(4))
print(f"AUC: {rmia_result['auc']:.4f}")
print(f"TPR@1%FPR: {rmia_result['tpr_at_1pct_fpr']:.4f}")
print(f"TPR@0.1%FPR: {rmia_result['tpr_at_0.1pct_fpr']:.4f}")

print('\n=== GLOBAL SUMMARY (Transformer + Attack) ===')
global_summary = pd.concat([
    transformer_metrics[['model', 'source', 'train_acc', 'test_acc', 'gap_acc', 'train_auc', 'test_auc', 'num_shadow_models', 'shadow_epochs']].copy(),
    attack_summary[['attack', 'auc', 'tpr_at_10pct_fpr', 'tpr_at_5pct_fpr', 'tpr_at_1pct_fpr', 'tpr_at_0.1pct_fpr', 'offline_a', 'num_reference_models', 'paired_tuning_auc']].copy(),
], axis=1)
display(global_summary.round(4))

=== ATTACK RMIA RESULTS ===


,attack,target_model_idx,num_reference_models,paired_tuning_auc,offline_a,auc,tpr_at_10pct_fpr,tpr_at_5pct_fpr,tpr_at_1pct_fpr,tpr_at_0.1pct_fpr,tpr_at_0pct_fpr,audit_samples,population_samples
0,RMIA_blackbox_enhanced,0,1,0.5943,0.1,0.588,0.0,0.0,0.0,0.0,0.0,354,107


AUC: 0.5880
TPR@1%FPR: 0.0000
TPR@0.1%FPR: 0.0000

=== GLOBAL SUMMARY (Transformer + Attack) ===


,model,source,train_acc,test_acc,gap_acc,train_auc,test_auc,num_shadow_models,shadow_epochs,attack,auc,tpr_at_10pct_fpr,tpr_at_5pct_fpr,tpr_at_1pct_fpr,tpr_at_0.1pct_fpr,offline_a,num_reference_models,paired_tuning_auc
0,target_transformer,"trained in notebook (risky profile, epochs=260)",1.0,0.9296,0.0704,1.0,0.9893,11,60,RMIA_blackbox_enhanced,0.588,0.0,0.0,0.0,0.0,0.1,1,0.5943


In [19]:
# Quick metrics snapshot for analysis
print('SNAPSHOT_TRANSFORMER')
print(transformer_metrics[['train_acc','test_acc','gap_acc','train_auc','test_auc','num_shadow_models','shadow_epochs']].iloc[0].to_dict())

print('SNAPSHOT_RMIA')
keys = ['auc','offline_a','num_reference_models','paired_tuning_auc','tpr_at_10pct_fpr','tpr_at_5pct_fpr','tpr_at_1pct_fpr','tpr_at_0.1pct_fpr','tpr_at_0pct_fpr']
print({k: rmia_result.get(k, None) for k in keys})

SNAPSHOT_TRANSFORMER
{'train_acc': 1.0, 'test_acc': 0.9295774647887324, 'gap_acc': 0.07042253521126762, 'train_auc': 1.0, 'test_auc': 0.9892803275156217, 'num_shadow_models': 11.0, 'shadow_epochs': 60.0}
SNAPSHOT_RMIA
{'auc': 0.5879778672032193, 'offline_a': 0.1, 'num_reference_models': 1, 'paired_tuning_auc': 0.5942896358006959, 'tpr_at_10pct_fpr': 0.0, 'tpr_at_5pct_fpr': 0.0, 'tpr_at_1pct_fpr': 0.0, 'tpr_at_0.1pct_fpr': 0.0, 'tpr_at_0pct_fpr': 0.0}
